In [64]:
from rdkit.Chem import RWMol
from rdkit import Chem
from rdkit.Chem import Draw
import torch
import networkx as nx
# from google.colab import drive
from torch_geometric.datasets import ZINC
from torch_geometric.data import Data
from IPython.display import display
import numpy as np
from sklearn.preprocessing import StandardScaler
import os
from sklearn.linear_model import Ridge, RidgeCV
import pandas as pd
from models.graphcnnVSA_Binding_FULL import GraphCNN
from sklearn.metrics import mean_squared_error,mean_absolute_error, r2_score

**Data Preprocess**

In [65]:


# Atom index to atomic number mapping (Based on PyG's dataset)
index_to_atomic_number = {0: 6, 1: 8, 2: 7, 3: 16, 4: 9}  # Carbon, Oxygen, Nitrogen, Sulfur, Fluorine

# Function to generate enhanced node features

def expand_atomic_features(data):
    num_nodes = data.x.shape[0]
    # print("data " , data)
    # print("x", data.x)
    # print("num nodes ", num_nodes)

    # Directly use atomic numbers if they are present
    atomic_numbers = data.x.reshape(-1, 1)  # No need for mapping

    # print("Atiomic numbers ", atomic_numbers)
    # print("data.edge_index -  ", data.edge_index)

    # Compute degree (number of bonds)
    degrees = torch.zeros((num_nodes, 1))
    for i in range(data.edge_index.shape[1]):
        u, v = data.edge_index[:, i]
        degrees[u] += 1
        degrees[v] += 1
    print("degrees ", degrees)

    # Additional atomic properties
    valence_electrons = torch.zeros((num_nodes, 1))
    hybridization = torch.zeros((num_nodes, 3))  # One-hot for (sp, sp2, sp3)
    aromaticity = torch.zeros((num_nodes, 1))

    valence_dict = {
        0:0,
        1: 1,    # Hydrogen (H)
        2: 2,    # Helium (He) - noble gas, usually inert
        3: 1,    # Lithium (Li)
        4: 2,    # Beryllium (Be)
        5: 3,    # Boron (B)
        6: 4,    # Carbon (C)
        7: 5,    # Nitrogen (N)
        8: 6,    # Oxygen (O)
        9: 7,    # Fluorine (F)
        10: 8,   # Neon (Ne) - noble gas
        11: 1,   # Sodium (Na)
        12: 2,   # Magnesium (Mg)
        13: 3,   # Aluminum (Al)
        14: 4,   # Silicon (Si)
        15: 5,   # Phosphorus (P)
        16: 6,   # Sulfur (S)
        17: 7,   # Chlorine (Cl)
        18: 8,   # Argon (Ar) - noble gas
        19: 1,   # Potassium (K)
        20: 2,   # Calcium (Ca)
        35: 7,   # Bromine (Br)
        53: 7,   # Iodine (I)
        30: 2,   # Zinc (Zn) - typically +2 oxidation
        29: 2,   # Copper (Cu) - commonly +1 or +2
        26: 2,   # Iron (Fe) - commonly +2, +3 (approximate)
        25: 2    # Manganese (Mn) - commonly +2 in simple cases
    }
    hybridization_dict = {
    1:  [0, 0, 1],  # Hydrogen (H) - sp3-like when bonded
    2:  [0, 0, 0],  # Helium (He) - noble gas, no bonding
    3:  [0, 0, 1],  # Lithium (Li) - single bond if bonded (ionic nature)
    4:  [0, 1, 0],  # Beryllium (Be) - linear (sp) but here considered as sp2 for simplicity
    5:  [0, 1, 1],  # Boron (B) - sp2 (BF3), sp3 in other cases
    6:  [1, 1, 1],  # Carbon (C) - sp, sp2, sp3
    7:  [0, 1, 1],  # Nitrogen (N) - sp2, sp3
    8:  [0, 1, 1],  # Oxygen (O) - sp2 (carbonyl), sp3 (alcohol)
    9:  [0, 0, 1],  # Fluorine (F) - typically sp3
    10: [0, 0, 0],  # Neon (Ne) - noble gas, no bonding
    11: [0, 0, 1],  # Sodium (Na) - ionic, but sp3-like if bonded
    12: [0, 0, 1],  # Magnesium (Mg) - sp3-like in complexes
    13: [0, 1, 1],  # Aluminum (Al) - sp2, sp3
    14: [0, 1, 1],  # Silicon (Si) - sp2, sp3
    15: [0, 1, 1],  # Phosphorus (P) - sp2, sp3
    16: [0, 1, 1],  # Sulfur (S) - sp2, sp3
    17: [0, 0, 1],  # Chlorine (Cl) - sp3
    18: [0, 0, 0],  # Argon (Ar) - noble gas, no bonding
    19: [0, 0, 1],  # Potassium (K) - ionic, sp3-like if bonded
    20: [0, 0, 1],  # Calcium (Ca) - ionic, sp3-like if bonded
    30: [0, 0, 1],  # Zinc (Zn) - sp3-like in complexes
    29: [0, 0, 1],  # Copper (Cu) - sp3-like in coordination
    26: [0, 0, 1],  # Iron (Fe) - sp3-like in coordination
    25: [0, 0, 1],  # Manganese (Mn) - sp3-like in coordination
    35: [0, 0, 1],  # Bromine (Br) - sp3
    53: [0, 0, 1],  # Iodine (I) - sp3
    }
    aromaticity_dict = {
    1: 0,    # Hydrogen (H)
    2: 0,    # Helium (He)
    3: 0,    # Lithium (Li)
    4: 0,    # Beryllium (Be)
    5: 0,    # Boron (B) - often part of π systems, but typically treated as non-aromatic in this context
    6: 1,    # Carbon (C) - Yes, in benzene, pyridine, etc.
    7: 1,    # Nitrogen (N) - Yes, in pyridine, imidazole
    8: 1,    # Oxygen (O) - Yes, in furan (but not always)
    9: 0,    # Fluorine (F)
    10: 0,   # Neon (Ne)
    11: 0,   # Sodium (Na)
    12: 0,   # Magnesium (Mg)
    13: 0,   # Aluminum (Al)
    14: 0,   # Silicon (Si)
    15: 0,   # Phosphorus (P) - rarely part of aromatic systems
    16: 1,   # Sulfur (S) - Yes, in thiophene
    17: 0,   # Chlorine (Cl)
    18: 0,   # Argon (Ar)
    19: 0,   # Potassium (K)
    20: 0,   # Calcium (Ca)
    30: 0,   # Zinc (Zn)
    29: 0,   # Copper (Cu)
    26: 0,   # Iron (Fe)
    25: 0,   # Manganese (Mn)
    35: 0,   # Bromine (Br)
    53: 0    # Iodine (I)
    }

    # Assign atomic properties
    for i, atomic_num in enumerate(atomic_numbers.squeeze().tolist()):
        valence_electrons[i] = valence_dict.get(int(atomic_num))

        # Hybridization one-hot encoding
        hybridization[i, :] = torch.tensor(hybridization_dict.get(int(atomic_num), [0,0,0]))

        # Aromaticity assumption
        aromaticity[i] = aromaticity_dict.get(int(atomic_num),0)


    
    enhanced_features = torch.cat((atomic_numbers, degrees, valence_electrons, hybridization, aromaticity), dim=1)
    # print("valence_electrons ", valence_electrons)
    # print("hybridization ", hybridization)
    # Concatenate all features
    # print("enhanced_features.shape : ",enhanced_features.shape)
    # print("atomic_numbers, degrees, valence_electrons, hybridization, aromaticity")
    # print("enhanced_features : ",enhanced_features)

    return enhanced_features

def pyg_graph_to_mol(data):
    """
    Convert a PyG Data object to an RDKit Mol object.
    """
    mol = RWMol()

    # Add atoms (assuming first feature in node feature vector is atomic number)
    for atom_feature in data.x:
        atomic_num = int(atom_feature[0])  # Extract atomic number from node features
        mol.AddAtom(Chem.Atom(atomic_num))

    # Convert tensors to numpy arrays
    edge_index = data.edge_index.numpy()
    edge_attr = data.edge_attr.numpy() if data.edge_attr is not None else None

    # Add bonds only if they don't exist
    for i in range(edge_index.shape[1]):  # Loop through edges
        start, end = int(edge_index[0, i]), int(edge_index[1, i])

        # Check if bond already exists
        if mol.GetBondBetweenAtoms(start, end) is None:
            # Determine bond type (modify this based on your dataset's edge attributes)
            bond_type = Chem.BondType.SINGLE  # Default to single bond
            mol.AddBond(start, end, bond_type)

    return mol

In [66]:
class S2VGraph(object):
  def __init__(self, g, label,mol, node_tags=None, node_features=None):
    '''
        g: a networkx graph
        label: an integer graph label
        node_tags: a list of integer node tags
        node_features: a torch float tensor, one-hot representation of the tag that is used as input to neural nets
        edge_mat: a torch long tensor, contain edge list, will be used to create torch sparse tensor
        neighbors: list of neighbors (without self-loop)
    '''
    self.label = label
    self.g = g
    self.node_tags = node_tags
    self.neighbors = []
    self.node_features = node_features
    self.edge_mat = 0
    self.mol = mol
    self.max_neighbor = 0

In [67]:
def create_nodetags(mol):
  # node tags for one mol

  feat_dict = {}
  node_tags = []

  # Create a NetworkX graph
  nx_graph = nx.Graph()

  # Add nodes (atoms)
  for i, atom in enumerate(mol.GetAtoms()):
      atom_symbol = atom.GetSymbol()
      nx_graph.add_node(i, label=atom_symbol)

  # Add edges (bonds)
  for bond in mol.GetBonds():
      start, end = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
      nx_graph.add_edge(start, end)

  node_tags = [atom.GetSymbol() for atom in mol.GetAtoms()]

  return nx_graph, node_tags

def create_g_list(dataset):
  g_list = []
  for data in dataset:
    mol = pyg_graph_to_mol(data)

    node_features = expand_atomic_features(data)
    # print("create_g_list : W ", node_features.shape)
    edge_index = data.edge_index
    edge_features = data.edge_attr #get_edge_index_and_features(mol)
    graph = Data(x=node_features, edge_index=edge_index, edge_attr=edge_features)

    g, node_tags = create_nodetags(mol)
    g_list.append(S2VGraph(g = g, label= data.y, mol= mol, node_tags= node_tags, node_features=torch.tensor(node_features, dtype=torch.float32)))
    break
  return g_list

In [68]:
import pandas as pd, torch
from torch_geometric.data import InMemoryDataset, Data
from rdkit import Chem

# Map RDKit bond types to a single integer (edge_attr is 1D like ZINC prints)
_BOND2ID = {
    Chem.BondType.SINGLE: 0,
    Chem.BondType.DOUBLE: 1,
    Chem.BondType.TRIPLE: 2,
    Chem.BondType.AROMATIC: 3,
}

def smiles_to_data(smi, yval):
    mol = Chem.MolFromSmiles(smi)
    if mol is None or mol.GetNumAtoms() == 0:
        return None

    # x: [num_nodes, 1] (long) — single integer per atom (e.g., atomic number)
    x = torch.tensor([[a.GetAtomicNum()] for a in mol.GetAtoms()], dtype=torch.long)

    # Edges (both directions) and 1D edge_attr (long)
    src, dst, eattr = [], [], []
    for b in mol.GetBonds():
        u, v = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        t = _BOND2ID.get(b.GetBondType(), 0)
        # add both directions
        src += [u, v]
        dst += [v, u]
        eattr += [t, t]

    edge_index = torch.tensor([src, dst], dtype=torch.long)
    edge_attr  = torch.tensor(eattr, dtype=torch.long)  # shape [E], same as ZINC print

    # y: [1] (float)
    y = torch.tensor([float(yval)], dtype=torch.float)

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)

class ZINCLikeCSV(InMemoryDataset):
    def __init__(self, csv_path, smiles_col="smiles_canon", target_col="LogS"):
        df = pd.read_csv(csv_path)
        super().__init__('.')
        graphs = []
        for smi, y in zip(df[smiles_col], df[target_col]):
            g = smiles_to_data(smi, y)
            if g is not None:
                graphs.append(g)
        self.data, self.slices = self.collate(graphs)

In [69]:
def load_zinc():

  # dataset_train = ZINC(root="data/ZINC", subset=True, split="train")  # Load ZINC dataset from PyG
  dataset_test_1 = ZINC(root="data/ZINC", subset=True, split="test")  # Load ZINC dataset from PyG
  # dataset_train = ZINC(root="data/ZINC", subset=True, split="train")  # Load ZINC dataset from PyG
  # print(dataset_test.head(10))
  dataset_test_2  = ZINCLikeCSV("final_data/final_unique_test.csv")
  dataset_train_2  = ZINCLikeCSV("final_data/final_unique_train.csv")
  # gl_train = create_g_list(dataset_train)
  # print(type(gl_train[1].node_features))
  # print(type(dataset_test_1))
  # print(dataset_test_1[0])
  # print(type(dataset_test_2))
  # print(dataset_test_2[0])


  gl_test = create_g_list(dataset_test_2)
  gl_train = create_g_list(dataset_test_2)
  # print(type(gl_train[1].node_features))
  return gl_train, gl_test #, gl_train, 

load_zinc()


[10:10:06] WARNING: not removing hydrogen atom without neighbors
[10:10:06] WARNING: not removing hydrogen atom without neighbors
[10:10:06] WARNING: not removing hydrogen atom without neighbors
[10:10:06] WARNING: not removing hydrogen atom without neighbors
[10:10:06] WARNING: not removing hydrogen atom without neighbors
[10:10:06] WARNING: not removing hydrogen atom without neighbors
[10:10:06] WARNING: not removing hydrogen atom without neighbors
[10:10:06] Explicit valence for atom # 5 N, 4, is greater than permitted
[10:10:06] WARNING: not removing hydrogen atom without neighbors
[10:10:06] Explicit valence for atom # 5 N, 4, is greater than permitted
[10:10:06] WARNING: not removing hydrogen atom without neighbors
[10:10:06] WARNING: not removing hydrogen atom without neighbors
[10:10:06] WARNING: not removing hydrogen atom without neighbors
[10:10:06] WARNING: not removing hydrogen atom without neighbors


degrees  tensor([[2.],
        [4.],
        [4.],
        [4.],
        [2.]])
degrees  tensor([[2.],
        [4.],
        [4.],
        [4.],
        [2.]])


C:\Users\22390013@students.ltu.edu.au\AppData\Local\Temp\ipykernel_2496\3101687279.py:36: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  g_list.append(S2VGraph(g = g, label= data.y, mol= mol, node_tags= node_tags, node_features=torch.tensor(node_features, dtype=torch.float32)))


([<__main__.S2VGraph at 0x1bbc52708e0>],
 [<__main__.S2VGraph at 0x1bba4437f40>])

VSA Conversion

In [70]:
def project_node_features(g_list, original_feature_dim, new_dim):
    # Set a random seed for reproducibility
    torch.manual_seed(0)
    # Generate a random projection matrix
    # R = np.random.randn(original_feature_dim, new_dim) / np.sqrt(new_dim)
    # Initialize a random weight matrix for projection
    W = torch.randn(original_feature_dim, new_dim) / np.sqrt(new_dim)
    print("W : ", W.shape)
    # Project node features for each graph
    for g in g_list:
        # Assuming g.node_features is a torch.Tensor
        if g.node_features is not None:
            # print(g.node_features)
            g.node_features  = torch.matmul(g.node_features, W)
            # print(g.node_features.shape)

    return g_list

In [71]:
def VSA_conversion(g_list, new_dim=None):
    # Add labels and edge_mat
    for g in g_list:
        g.neighbors = [[] for _ in range(len(g.g))]

        # Build neighbors list
        for i, j in g.g.edges():
            g.neighbors[i].append(j)
            g.neighbors[j].append(i)

        # Compute max degree
        degree_list = [len(g.neighbors[i]) for i in range(len(g.g))]
        g.max_neighbor = max(degree_list)

        # Create edge matrix
        edges = [list(pair) for pair in g.g.edges()]
        edges.extend([[j, i] for i, j in edges])
        g.edge_mat = torch.LongTensor(edges).transpose(0, 1)

    #Extracting unique tag labels
    # tagset = set([])
    # for g in g_list:
    #     tagset = tagset.union(set(g.node_tags))

    # tagset = list(tagset)
    # tag2index = {tagset[i]:i for i in range(len(tagset))}


    ########## This part make one hit encoding of each node as they contain different atoms
    # for g in g_list:
    #     g.node_features = torch.zeros(len(g.node_tags), len(tagset))
    #     g.node_features[range(len(g.node_tags)), [tag2index[tag] for tag in g.node_tags]] = 1
            # hypervector[range(len(g.node_tags)), [tag2index[tag] for tag in node_tags if tag in tag2index]] = 1

    original_feature_dim = len(g_list[0].node_features[0])# len(tagset)
    # print(len(tagset))
    print("VSA_conversion",len(g_list[0].node_features[0]))


    if new_dim:
        g_list = project_node_features(g_list, original_feature_dim, new_dim)
    return g_list

**Make Embeddings**

In [72]:
delta_eq1 = 1
delta_eq2 = 2
delta_eq3 = 1
delta_eq4 = 2
equation_eq1 = 10
equation_eq2 = 10
equation_eq3 = 11
equation_eq4 = 11

In [73]:
def getEmbedding( model, device, train_graphs, batch_size=100, SUM = True):

    model.to(device)
    model.train()

    combined_embeddings = []  # Initialize the total embedding
    all_labels = []

    # Create batches
    num_graphs = len(train_graphs)
    for start_idx in range(0, num_graphs, batch_size):
        end_idx = min(start_idx + batch_size, num_graphs)
        batch_graphs = train_graphs[start_idx:end_idx]
        # print("getEmbedding :: Before BBBBB")
        output = model(batch_graphs)

        ################### For regression taskuse TRUE and false both as sum ################

        '''
        if(SUM==True):    # allways use SUM
            # Sum all embeddings
            combined_embedding = torch.sum(torch.stack(output), dim=0)  # Sum along the new batch dimension
            # 100 tensors
        else:
            #Concat
            combined_embedding = torch.cat(output, dim=1)
        '''
        # Add the summed embeddings of this batch to the total embedding
        #############################################################################
        combined_embeddings.append(output) #combined_embedding)

        ####################################Place conmvert label########
        # Collect labels
        labels = torch.FloatTensor([graph.label for graph in batch_graphs]).to(device)
        all_labels.append(labels)


    final_labels = torch.cat(all_labels, dim=0)
    final_embeddings = torch.cat(combined_embeddings, dim=0)

    # print("getEmbedding :: endo")
    return final_embeddings, final_labels

**Regression**

In [74]:

def ridge_regression(X_train, y_train, alpha=0.0001):

    # Z_train = ss.fit_transform(X_train)

    # Initialize Ridge Regression model
    # ridge_model = Ridge(alpha=alpha)

     # list of alphas to check: 100 values from 0 to 5 with
    r_alphas = np.logspace(0, 5, 100)

    # initiate the cross validation over alphas
    ridge_model = RidgeCV(alphas=r_alphas, scoring='r2')

    # Fit the model on training data
    ridge_model.fit(X_train, y_train)

    # Predict on the test set
    # y_pred = ridge_model.predict(X_test)

    return ridge_model

**evaluation**

In [75]:
file_path = "Results/summary_sol_4_CV_xFeatures.txt"
file = open(file_path, "a+")
file.write(f"\nDataset,Dimention,K-Hop,GVFA eq,MAE")
file.close()
def evaluate_ridge_predictions(model, test_x, test_y, dim, d_name, GVFA, hop):
    file = open(file_path, "a")
    all_predictions = []
    all_labels = []
    ndcg_scores = []
    spearman_correlations = []

    predicted_y = model.predict(test_x)
    # print("test_y :", test_y)
    # print("predicted_y :", predicted_y)
    mae = mean_absolute_error(test_y, predicted_y)
    # mse = mean_squared_error(test_y, predicted_y)
    print( "Data - ", d_name," Dimention - ", dim," k-hop - ", hop, " GVFA - ", GVFA, " MAE - ", mae)
    file.write(f"\n{d_name},{dim},{hop},{GVFA},{mae}")
    file.close()

**Neural Network**

In [76]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SolubilityMLP(nn.Module):
    def __init__(self, input_dim=2000, hidden_dim=512, hidden_dim2=256, output_dim=1):
        super(SolubilityMLP, self).__init__()

        # Input to first hidden layer
        self.fc1 = nn.Linear(input_dim, hidden_dim)

        # First hidden to second hidden layer
        self.fc2 = nn.Linear(hidden_dim, hidden_dim2)

        # Second hidden to output layer
        self.fc3 = nn.Linear(hidden_dim2, output_dim)  # Regression output

        # Optional: Dropout for regularization
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):  # x: [batch_size, 2000]
        x = F.relu(self.fc1(x))
        x = self.dropout(x)  # Prevent overfitting

        x = F.relu(self.fc2(x))
        x = self.dropout(x)  # Prevent overfitting

        output = self.fc3(x)  # [batch_size, 1]
        return output

def NN_main(embeddings, labels):
  # Example setup
  input_dim = 2000
  hidden_dim = 512
  hidden_dim2 = 256
  output_dim = 1

  # Initialize model
  model = SolubilityMLP(input_dim, hidden_dim, hidden_dim2, output_dim)
  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
  model = model.to(device)

  # Loss function and optimizer
  criterion = nn.MSELoss()  # For regression
  optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

  epochs = 50  # Number of training epochs

  for epoch in range(epochs):
      model.train()
      total_loss = 0

      for batch_data, batch_targets in train_loader:  # Assuming you have a DataLoader
          batch_data, batch_targets = batch_data.to(device), batch_targets.to(device)

          optimizer.zero_grad()

          output = model(batch_data)  # [batch_size, 1]
          loss = criterion(output.squeeze(), batch_targets.squeeze())  # Compute MSE Loss
          loss.backward()
          optimizer.step()

          total_loss += loss.item() * batch_data.size(0)  # Sum loss for the batch

      avg_loss = total_loss / len(train_loader.dataset)
      print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")

In [77]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch_data, batch_targets in loader:
            batch_data, batch_targets = batch_data.to(device), batch_targets.to(device)
            output = model(batch_data)
            loss = criterion(output.squeeze(), batch_targets.squeeze())
            total_loss += loss.item() * batch_data.size(0)
    return total_loss / len(loader.dataset)

**Main **

In [78]:


d_name = "zinc"

new_dim = [200, 500, 1000, 5000, 10000, 20000, 30000, 40000, 50000, 100000]
data_name = ["zinc"] # , "mpo"
num_layer_list = [1,2,3,4,5]


for dim in new_dim:
  for num_layers in num_layer_list:

    gl_train, gl_test = load_zinc()
    gl_VSA_train = VSA_conversion(gl_train, dim)
    gl_VSA_test = VSA_conversion(gl_test, dim)

    graph_pooling_type = 'sum'  # sum, average
    neighbor_pooling_type = 'sum' # sum, average, max
    device = 1  # help='if delta is 1 will be the model with binding, if 0 model will have be without binding (default: 1)'
    device = torch.device('cpu')

    model_eq1 = GraphCNN(gl_VSA_train[0].node_features.shape[1], num_layers, delta_eq1, graph_pooling_type, neighbor_pooling_type, device, equation_eq1) #.to(device)
    # model_eq2 = GraphCNN(gl_VSA_train[0].node_features.shape[1], num_layers, delta_eq2, graph_pooling_type, neighbor_pooling_type, device, equation_eq2) #.to(device)
    # model_eq3 = GraphCNN(gl_VSA_train[0].node_features.shape[1], num_layers, delta_eq3, graph_pooling_type, neighbor_pooling_type, device, equation_eq3) #.to(device)
    # model_eq4 = GraphCNN(gl_VSA_train[0].node_features.shape[1], num_layers, delta_eq4, graph_pooling_type, neighbor_pooling_type, device, equation_eq4) #.to(device)

    # here getEmbedding functuon change labels
    embeddings_eq1, labels_eq1 = getEmbedding(model_eq1, device, gl_VSA_train)
    test_embeddings_eq1, test_labels_eq1 = getEmbedding(model_eq1, device, gl_VSA_test)

    # embeddings_eq2, labels_eq2 = getEmbedding(model_eq2, device, gl_VSA_train)
    # test_embeddings_eq2, test_labels_eq2 = getEmbedding(model_eq2, device, gl_VSA_test)

    # embeddings_eq3, labels_eq3 = getEmbedding(model_eq3, device, gl_VSA_train)
    # test_embeddings_eq3, test_labels_eq3 = getEmbedding(model_eq3, device, gl_VSA_test)

    # embeddings_eq4, labels_eq4 = getEmbedding(model_eq4, device, gl_VSA_train)
    # test_embeddings_eq4, test_labels_eq4 = getEmbedding(model_eq4, device, gl_VSA_test)


    # regression

    scaler = StandardScaler(with_mean=False)

    ridge_model_1 = ridge_regression(embeddings_eq1, labels_eq1)
    # ridge_model_2 = ridge_regression(embeddings_eq2, labels_eq2)
    # ridge_model_3 = ridge_regression(embeddings_eq3, labels_eq3)
    # ridge_model_4 = ridge_regression(embeddings_eq4, labels_eq4)

    # print(np.array_equal(ridge_model_1.coef_, ridge_model_2.coef_))  # ✅ True if same
    # print(np.array_equal(ridge_model_1.intercept_, ridge_model_2.intercept_))  # ✅ True if same


    evaluate_ridge_predictions(ridge_model_1, test_embeddings_eq1, test_labels_eq1, dim, d_name, 1, num_layers)
    # evaluate_ridge_predictions(ridge_model_2, test_embeddings_eq2, test_labels_eq2, dim, d_name, 2, num_layers)
    # evaluate_ridge_predictions(ridge_model_3, test_embeddings_eq3, test_labels_eq3, dim, d_name, 3, num_layers)
    # evaluate_ridge_predictions(ridge_model_4, test_embeddings_eq4, test_labels_eq4, dim, d_name, 4, num_layers)

    del model_eq1
    # del model_eq2
    # del model_eq3
    # del model_eq4

    # del ridge_model_1
    # del ridge_model_2
    # del ridge_model_3
    # del ridge_model_4
    del gl_test
    del gl_train
    del gl_VSA_train
    del gl_VSA_test
    # del g_list_VSA

    del embeddings_eq1
    # del embeddings_eq2
    # del embeddings_eq3
    # del embeddings_eq4

    del labels_eq1
    # del labels_eq2
    # del labels_eq3
    # del labels_eq4


[10:10:13] WARNING: not removing hydrogen atom without neighbors
[10:10:13] WARNING: not removing hydrogen atom without neighbors
[10:10:13] WARNING: not removing hydrogen atom without neighbors
[10:10:13] WARNING: not removing hydrogen atom without neighbors
[10:10:13] WARNING: not removing hydrogen atom without neighbors
[10:10:13] WARNING: not removing hydrogen atom without neighbors
[10:10:14] WARNING: not removing hydrogen atom without neighbors
[10:10:14] Explicit valence for atom # 5 N, 4, is greater than permitted
[10:10:14] WARNING: not removing hydrogen atom without neighbors
[10:10:14] Explicit valence for atom # 5 N, 4, is greater than permitted
[10:10:14] WARNING: not removing hydrogen atom without neighbors
[10:10:14] WARNING: not removing hydrogen atom without neighbors
[10:10:14] WARNING: not removing hydrogen atom without neighbors
[10:10:14] WARNING: not removing hydrogen atom without neighbors


degrees  tensor([[2.],
        [4.],
        [4.],
        [4.],
        [2.]])
degrees  tensor([[2.],
        [4.],
        [4.],
        [4.],
        [2.]])
VSA_conversion 7
W :  torch.Size([7, 200])
VSA_conversion 7
W :  torch.Size([7, 200])
Input feature size:  200


C:\Users\22390013@students.ltu.edu.au\AppData\Local\Temp\ipykernel_2496\3101687279.py:36: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  g_list.append(S2VGraph(g = g, label= data.y, mol= mol, node_tags= node_tags, node_features=torch.tensor(node_features, dtype=torch.float32)))


ValueError: Found array with dim 3. _RidgeGCV expected <= 2.

In [79]:
df = pd.read_csv("final_data/final_unique_train.csv")
df.loc[13158, "smiles_canon"] = "Cc1cc[n+](O)cc1"
df.loc[13186, "smiles_canon"] = "O=C(O)c1cc[n+](O)cc1"
df.to_csv("final_data/final_unique_train_fixed.csv", index=False)